In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv('data//1_preprocessed//preprocessed_dataset.csv')

y = data.track_genre
X = data[['duration_ms','explicit','danceability','energy','key','loudness','mode','speechiness','acousticness','instrumentalness','liveness','valence','tempo','time_signature']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Categorical features (will want to one-hot encode these): 'key', 'mode', 'time_signature'
categorical_cols = ['key', 'mode', 'time_signature']
print("Cardinality of categorical columns:")
for col in categorical_cols:
    print(f"{col}: {X[col].nunique()}")
# Only key (12) may have too high of a cardinality, but we can still encode it to see how it performs

numerical_cols = ['duration_ms','explicit','danceability','energy','loudness','speechiness','acousticness','instrumentalness','liveness','valence','tempo']
print("Numerical columns:", numerical_cols)


Cardinality of categorical columns:
key: 12
mode: 2
time_signature: 5
Numerical columns: ['duration_ms', 'explicit', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'  # Keep the numerical columns as they are; already preprocessed
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Candidate hyperparameters to test (find balance between over and underifitting, while still training in a reasonable amount of time)
candidate_params = [
    {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1},
    {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1},
    {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1},
    {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 5, 'min_samples_leaf': 2},
]

results = []

for params in candidate_params:
    model = RandomForestClassifier(random_state=0, verbose=True, **params)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1)

    results.append({
        'params': params,
        'mean_cv_accuracy': scores.mean(),
        'std_cv_accuracy': scores.std()
    })

results = pd.DataFrame(results).sort_values('mean_cv_accuracy', ascending=False)
print(results)

best_params = results.iloc[0]['params']
print("Best hyperparameters:")
print(best_params)


KeyboardInterrupt: 

In [ ]:
# Best parameters found, train the final model using those hyperparameters
best_model = RandomForestClassifier(random_state=0, verbose=True, **best_params)
my_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', best_model)
])

my_pipeline.fit(X_train, y_train)

preds = my_pipeline.predict(X_test)

print("Best hyperparameters:")
print(best_params)


Fitting 3 folds for each of 24 candidates, totalling 72 fits


KeyboardInterrupt: 

In [25]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

preds = my_pipeline.predict(X_test)

cm = confusion_matrix(y_test, preds)
print("Confusion Matrix:")
print(cm)


print("Classification Report:")
print(classification_report(y_test, preds))

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.9s


Confusion Matrix:
[[35  0  0 ...  1  1  4]
 [ 0 53  0 ...  7  1  0]
 [ 0  1 10 ...  0  1  6]
 ...
 [ 1 10  0 ... 44  1  1]
 [ 2  2  1 ...  3 17  1]
 [ 2  0  3 ...  0  3 67]]
Classification Report:
                   precision    recall  f1-score   support

         acoustic       0.18      0.17      0.17       210
         afrobeat       0.32      0.26      0.29       204
         alt-rock       0.06      0.05      0.05       208
      alternative       0.16      0.13      0.14       207
          ambient       0.20      0.17      0.18       204
            anime       0.14      0.09      0.11       191
      black-metal       0.40      0.49      0.44       194
        bluegrass       0.29      0.43      0.34       195
            blues       0.14      0.11      0.12       198
           brazil       0.01      0.01      0.01       189
        breakbeat       0.29      0.24      0.26       198
          british       0.08      0.05      0.06       235
         cantopop       0.20      0

[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    1.9s finished
